# Final experiment: structured posterior coverage-aware SNIS allocation
**Frozen follow-up protocol — do not tune on the test results.**

This study keeps the original SNIS estimator, fixed proposals, sample budget, and coverage-risk representation. The only methodological changes are coherent prediction of posterior region masses and conditional means, and validation-selected penalized correction toward an inexpensive allocation. It tests whether the earlier privileged result can be converted into a nonprivileged policy.

For region $A_j$, let $\pi_j=p(A_j)$, $\rho_j=q(A_j)$, $m_j=E_p[z\mid A_j]$. The structured predictor supplies $\hat\pi_j$ and $\hat m_j$, with $\hat\mu=\sum_j\hat\pi_j\hat m_j$. Reconstruct
$$\hat b_j=\frac{1}{d}\left\|\hat\mu-\frac{\sum_{\ell\ne j}\hat\pi_\ell\hat m_\ell}{\sum_{\ell\ne j}\hat\pi_\ell}\right\|^2,\qquad \hat R_i(K)=\hat a_i/K+\gamma\sum_j\hat b_{ij}(1-\rho_{ij})^K.$$
The coverage expression is a surrogate, not an exact additive decomposition of SNIS risk. Multiple missed regions can overlap as events. The allocation minimizes the surrogate plus a convex baseline-deviation penalty, with exact integer budgets. No posterior evaluations are used to select the deployable allocation online. The posterior student is trained offline on exact posterior labels, which is privileged training information; this study does not establish that such labels are cheap to obtain in arbitrary models.

**Primary confirmatory suite:** new synthetic cohorts with seeds disjoint from the previous notebooks; independent training, risk-teacher, validation, oracle A/B and evaluation banks. Six prespecified settings, paired batch-level inference, Holm correction across the two primary contrasts in all six settings. **Real-data follow-up:** official UJIIndoorLoc test records have already been examined in earlier work. They remain exploratory, even with independent MC streams; there is no untouched external real-data cohort in this notebook. No further tuning on these records. The model is fitted using only official training data.

The primary deployable method must improve by at least 2% over the validation-selected cheap policy, beat the validation-selected matched-feature generic learned policy, have positive evidence for the no-coverage ablation, and capture at least 25% of positive estimated oracle headroom. Report all failures, not just favorable cases. Runtime vs exact posterior and direct posterior-student prediction is a separate practical criterion. A validation-selected fallback is a valid outcome but not a successful improvement.

Run all cells once with GPU (CPU also supported). Default is the complete experiment. Outputs are saved as they are produced; the final ZIP includes the raw statistics, fitted models and the protocol manifest. If a download fails, upload the official UCI ZIP when prompted. The experiment does not need earlier notebook files.

Component ablations are evaluated both at the validation-selected penalty and in the unpenalized policy, so a fallback does not hide the mass/moment estimation diagnostic. All policies are checked for exact budget and allocation bounds.

In [ ]:
# Install only missing dependencies (standard Colab usually has all of these).
import importlib.util, subprocess, sys
for module, package in [('numba','numba'),('sklearn','scikit-learn'),('scipy','scipy'),('joblib','joblib'),('pandas','pandas'),('matplotlib','matplotlib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

In [ ]:
import os,io,zipfile,urllib.request,json,time,hashlib,math,shutil,platform
os.environ.setdefault("OMP_NUM_THREADS","4")
os.environ.setdefault("MKL_NUM_THREADS","4")
from pathlib import Path
import numpy as np,pandas as pd,torch,matplotlib.pyplot as plt
from torch import nn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from scipy.stats import t,spearmanr,binom
from scipy.special import ndtr,logsumexp
from numba import njit
from IPython.display import display
import joblib,itertools
FULL=True
SEED=20261123  # same fitted Wi-Fi pipeline as previous benchmark
NEW=932011  # new sampling streams / synthetic cohorts
TEST_FIXTURE=False  # local implementation test only; never set for real study
KMIN,KMAX,KAVG=4,120,48; KS=np.arange(KMIN,KMAX+1); BATCH=32
NOPT=512 if FULL else 12; NEVAL=1024 if FULL else 16; NTEACH=512 if FULL else 12; NVAL=512 if FULL else 12
EPOCHS=30 if FULL else 2
OUT=Path('/content' if Path('/content').exists() else '/tmp')/('final_structured_coverage_'+time.strftime('%Y%m%d_%H%M%S'));OUT.mkdir(exist_ok=False)
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.manual_seed(SEED);np.random.seed(SEED)
if DEV.type=='cuda':torch.cuda.manual_seed_all(SEED)
else:torch.set_num_threads(4)
torch.backends.cuda.matmul.allow_tf32=False
start_total=time.perf_counter()
def T(x,dtype=torch.float32):return torch.as_tensor(x,dtype=dtype,device=DEV)
def sync():
    if DEV.type=='cuda':torch.cuda.synchronize()
print('Device:',DEV,'FULL:',FULL)


## 1. Mathematical and optimizer validation

In [ ]:
def two_point(K,pi,rho):
    j=np.arange(K+1); w1=pi/rho;w0=(1-pi)/(1-rho)
    est=j*w1/(j*w1+(K-j)*w0)
    exact=np.sum(binom.pmf(j,K,rho)*(est-pi)**2)
    coverage=pi*pi*(1-rho)**K+(1-pi)**2*rho**K
    return exact,coverage
for pi in [.1,.5,.9]:
    for rho in [.01,.2,.5,.99]:
        a,b=two_point(1,pi,rho);assert np.isclose(a,b)
        for k in [4,48,120]:
            a,b=two_point(k,pi,rho);assert a+1e-12>=b
for k in [4,48,120]:
    a,_=two_point(k,.3,.3);assert np.isclose(a,.3*.7/k)
print('Exact binomial identity checks passed.')

pi0=np.array([[.3,.7]]); cm0=np.array([[[1.],[0.]]]);

In [ ]:
@njit
def dp(risk,budget,kmin):
    n,L=risk.shape;extra=budget-n*kmin;mx=L-1;prev=np.full(extra+1,np.inf);prev[0]=0.;back=np.full((n,extra+1),-1,np.int32)
    for i in range(n):
        cur=np.full(extra+1,np.inf)
        for b in range(max(0,extra-(n-i-1)*mx),min(extra,(i+1)*mx)+1):
            for k in range(max(0,b-i*mx),min(mx,b)+1):
                z=prev[b-k]+risk[i,k]
                if z<cur[b]:cur[b]=z;back[i,b]=k
        prev=cur
    out=np.empty(n,np.int32);b=extra
    for i in range(n-1,-1,-1):k=back[i,b];out[i]=k+kmin;b-=k
    return out
import itertools
sm=np.array([[7.,4.,5.],[2.,6.,1.],[3.,1.,8.]]);k=dp(sm,5,1);assert sum(k)==5 and np.isclose(sum(sm[i,j-1] for i,j in enumerate(k)),min(sum(sm[i,j-1] for i,j in enumerate(z)) for z in itertools.product(range(1,4),repeat=3) if sum(z)==5))
def batches(n,seed):
    a=np.random.default_rng(seed).permutation(n);return [a[i:i+BATCH] for i in range(0,n,BATCH)]
@njit
def greedy(r,budget,kmin):
    n,L=r.shape;k=np.zeros(n,np.int32)
    for _ in range(budget-n*kmin):
        best=-1;gain=-np.inf
        for i in range(n):
            j=k[i]
            if j<L-1:
                z=r[i,j]-r[i,j+1]
                if z>gain:best=i;gain=z
        k[best]+=1
    return k+kmin
# Greedy is exact for diminishing marginal gains. Otherwise use unrestricted DP.
# This avoids charging the cheap baseline for an unnecessarily slow optimizer.
def alloc(r,bs):
    out=np.zeros(len(r),np.int32)
    for ix in bs:
        rr=np.ascontiguousarray(r[ix],dtype=np.float64);g=rr[:,:-1]-rr[:,1:]
        solver=greedy if np.all(g[:,:-1]>=g[:,1:]) else dp
        out[ix]=solver(rr,len(ix)*KAVG,KMIN)
    assert np.all((out>=KMIN)&(out<=KMAX)) and all(out[ix].sum()==len(ix)*KAVG for ix in bs);return out


## 2. Structured prediction and baseline-aware allocation
The penalty is $\lambda s_{batch}\sum_i |K_i-K_i^0|$, where the scale is the median predicted risk at 48 draws divided by 48. It preserves diminishing marginal gains for the structured surrogate. Generic comparators receive the same predicted posterior information and the same penalty menu. Validation selects all settings, including a baseline-only fallback.

In [ ]:
# Shared structured region prediction, risk curves and validation-only policy selection.
from scipy.stats import t as student_t
from sklearn.ensemble import HistGradientBoostingRegressor
FLOOR=1e-10
GAMMAS=[.1,.25,.5,1.,2.]
PENALTIES=[0.,.25,1.,4.]
KNOTS=np.array([4,8,16,24,32,48,64,80,96,120]); KI=KNOTS-KMIN

def reg(seed):return HistGradientBoostingRegressor(max_iter=90 if FULL else 5,max_leaf_nodes=15,min_samples_leaf=15 if FULL else 3,l2_regularization=2,random_state=seed)

def coefficients(pi,cm):
    # Conditional means cm and masses pi determine a coherent global mean.
    pi=np.asarray(pi,np.float64);cm=np.asarray(cm,np.float64);pi=pi/pi.sum(1,keepdims=True)
    mu=(pi[:,:,None]*cm).sum(1);n,J=pi.shape;d=cm.shape[-1];b=np.zeros_like(pi)
    for j in range(J):
        mask=np.arange(J)!=j; p=pi[:,mask]; denom=p.sum(1); outside=(p[:,:,None]*cm[:,mask]).sum(1)/np.maximum(denom[:,None],1e-300)
        b[:,j]=np.where(denom>1e-14,((mu-outside)**2).mean(1),0.)
    assert np.isfinite(b).all() and (b>=0).all();return b

def cover(x,b):return np.einsum('nj,njk->nk',b,np.power(1-np.clip(x['rho'],0,1)[:,:,None],KS[None,None,:]))

def pack(F,H,rho,b):
    F=np.asarray(F,float);H=np.asarray(H,float);rho=np.asarray(rho,float);b=np.asarray(b,float)
    assert np.allclose(rho.sum(1),1,atol=1e-5) and np.isfinite(H).all() and np.isfinite(F).all() and np.isfinite(b).all()
    extra=np.column_stack([-np.sum(rho*np.log(np.maximum(rho,FLOOR)),1),np.sum(rho<.02,1),np.sum(rho**2,1)])
    F=np.column_stack([F,extra,H.reshape(len(H),-1)]);return dict(F=F,H=H,rho=rho,b=np.maximum(b,0))

def augment(x,pi,cm):
    # Identical student-derived inputs for all generic learned risk models.
    b=coefficients(pi,cm);C=cover(x,b);mean=(pi[:,:,None]*cm).sum(1)
    F=np.column_stack([x['F'],pi, (cm-mean[:,None,:]).reshape(len(pi),-1),b,C[:,KI]])
    return dict(x,F=F,pred_pi=pi,pred_cm=cm,pred_b=b)

def train_regions(x,pi,cm,seed):
    # Synthetic-only structured student. Train on a separate distillation cohort.
    X=x['F'];center=X.mean(0);scale=X.std(0);scale[scale<1e-6]=1
    # Standardize means separately by region/dimension using the training cohort only.
    c0=cm.mean(0);cs=cm.std(0);cs[cs<1e-5]=1
    J,d=cm.shape[1:];torch.manual_seed(seed)
    net=nn.Sequential(nn.Linear(X.shape[1],192),nn.LayerNorm(192),nn.SiLU(),nn.Linear(192,128),nn.SiLU(),nn.Linear(128,J*(d+1))).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=.001,weight_decay=.0001);xt=T((X-center)/scale);pt=T(pi);ct=T((cm-c0)/cs)
    for ep in range(50 if FULL else 2):
        net.train()
        for ii in torch.randperm(len(X),device=DEV).split(128):
            pred=net(xt[ii]);logp=pred[:,:J].log_softmax(1);mc=pred[:,J:].reshape(-1,J,d)
            # Small positive weights retain supervision for uncommon regions.
            w=pt[ii]+.02/J;loss=-(pt[ii]*logp).sum(1).mean()+.5*(w*((mc-ct[ii])**2).mean(2)).sum(1).mean()
            opt.zero_grad();loss.backward();opt.step()
    net.eval();return net,center,scale,c0,cs,J,d

@torch.no_grad()
def region_predict(m,x):
    net,center,scale,c0,cs,J,d=m;z=net(T((x['F']-center)/scale));pi=z[:,:J].softmax(1).double().cpu().numpy();cm=z[:,J:].reshape(-1,J,d).double().cpu().numpy()*cs+c0
    return pi,cm

def amplitude(R,C,g):
    # Fit only the inverse-K amplitude, with the coverage shape held fixed.
    w=1/np.maximum(R,np.maximum(np.median(R,1,keepdims=True)*.05,FLOOR))**2
    return np.maximum(np.sum(w*(R-g*C)/KS,axis=1)/np.sum(w/KS**2,axis=1),FLOOR)

def inv_predict(m,F):return np.exp(np.clip(m.predict(F),-30,30))[:,None]/KS

def fit_risk(x,R,seed):
    F=x['F'];C=cover(x,x['pred_b']);Y=np.log(np.maximum(R,FLOOR));logk=np.log(KS/KAVG);D=np.stack([np.ones_like(logk),-logk],1);pa=Y@np.linalg.pinv(D).T;pa[:,1]=np.clip(pa[:,1],.05,2.5);pa[:,0]=np.mean(Y+pa[:,1,None]*logk,1)
    inv=reg(seed).fit(F,np.log(amplitude(R,C,0)))
    power=[reg(seed+1+j).fit(F,pa[:,j]) for j in range(2)]
    direct=[reg(seed+10+j).fit(F,Y[:,i]) for j,i in enumerate(KI)]
    amps={g:reg(seed+30+j).fit(F,np.log(amplitude(R,C,g))) for j,g in enumerate(GAMMAS)}
    # Direct coefficient regression comparator, using the same input features.
    from sklearn.model_selection import KFold
    def fit_b(ix):
        b=x['b'][ix];s=max(np.median(b[b>0]) if np.any(b>0) else 1.,FLOOR);h=region_features(x,ix);y=np.log1p(b.ravel()/s);return reg(seed+70).fit(h,y),s,float(y.max())
    def pred_b(m,ix):
        rr,s,cap=m;return (s*np.expm1(np.clip(rr.predict(region_features(x,ix)),0,cap+1))).reshape(len(ix),-1)
    oof=np.zeros_like(x['b']);allix=np.arange(len(F))
    for tr,va in KFold(3,shuffle=True,random_state=seed).split(F):oof[va]=pred_b(fit_b(tr),va)
    dc=cover(x,oof);damps={g:reg(seed+80+j).fit(F,np.log(amplitude(R,dc,g))) for j,g in enumerate(GAMMAS)}
    return dict(inv=inv,power=power,direct=direct,amps=amps,b=fit_b(allix),damps=damps)

def region_features(x,ix):
    H=x['H'][ix];F=x['F'][ix];n,J=H.shape[:2];return np.concatenate([H,np.broadcast_to(F[:,None,:],(n,J,F.shape[1]))],2).reshape(n*J,-1)

def direct_b(m,x):
    rr,s,cap=m['b'];return (s*np.expm1(np.clip(rr.predict(region_features(x,np.arange(len(x['F'])))),0,cap+1))).reshape(x['rho'].shape)

def curves(m,x):
    F=x['F'];p=m['power'];z=np.exp(np.clip(p[0].predict(F)[:,None]-np.clip(p[1].predict(F),.05,2.5)[:,None]*np.log(KS/KAVG),-30,30));lg=np.stack([r.predict(F) for r in m['direct']],1);flex=np.exp(np.clip(np.stack([np.interp(np.log(KS),np.log(KNOTS),r) for r in lg]),-30,30));C=cover(x,x['pred_b']);D=cover(x,direct_b(m,x))
    out={'inverse_K':inv_predict(m['inv'],F),'power_law':z,'direct_risk':flex}
    out.update({f'structured_{g:g}':inv_predict(a,F)+g*C for g,a in m['amps'].items()});out.update({f'direct_b_{g:g}':inv_predict(a,F)+g*D for g,a in m['damps'].items()});return out

def corrected(r,base,pen,bs):
    if pen=='fallback':return base.copy()
    r=r.copy()
    for ix in bs:
        # Convex integer penalty toward the validation-selected cheap policy.
        scale=max(float(np.median(r[ix,KAVG-KMIN]))/KAVG,FLOOR)
        r[ix]+=float(pen)*scale*abs(KS[None,:]-base[ix,None])
    return alloc(r,bs)

def value(R,k):return float(R[np.arange(len(k)),k-KMIN].mean())

def choose(pred,cheap,R,bs):
    cp={k:alloc(r,bs) for k,r in cheap.items()};base=min(cp,key=lambda k:value(R,cp[k]));b=cp[base];rows=[]
    for k,v in pred.items():
        for p in PENALTIES+['fallback']:
            counts=corrected(v,b,p,bs);rows.append(dict(curve=k,penalty=p,mse=value(R,counts)))
    def best(rows):return min(rows,key=lambda a:a['mse'])
    # Independently tune all comparators on identical validation data.
    sel={'base':base,'structured':best([r for r in rows if r['curve'].startswith('structured_')]),'direct_b':best([r for r in rows if r['curve'].startswith('direct_b_')]),'generic':best([r for r in rows if r['curve'] in ['inverse_K','power_law','direct_risk']]),'no_coverage':best([r for r in rows if r['curve']=='inverse_K']),'structured_raw':best([r for r in rows if r['curve'].startswith('structured_') and r['penalty']==0.])}
    return sel,pd.DataFrame(rows)

def policies(m,x,cheap,bs,sel,RA,RB,tau):
    pred=curves(m,x);P={k:alloc(v,bs) for k,v in cheap.items()};base=P[sel['base']];P['cheap_selected']=base.copy()
    for name in ['structured','direct_b','generic','no_coverage','structured_raw']:
        z=sel[name];P[name]=corrected(pred[z['curve']],base,z['penalty'],bs)
    # Pure component substitutions, same frozen amplitude and gamma; privileged diagnostic only.
    g=float(sel['structured']['curve'].split('_')[-1]);amp=inv_predict(m['amps'][g],x['F']);pen=sel['structured']['penalty'];
    for name,pi,cm in [('true_mass',x['pi'],x['pred_cm']),('true_moments',x['pred_pi'],x['cm']),('true_b',x['pi'],x['cm'])]:
        P['ablation_'+name]=corrected(amp+g*cover(x,coefficients(pi,cm)),base,pen,bs)
    # Also isolate errors in the unpenalized policy, even if the deployed choice is fallback.
    raw_g=float(sel['structured_raw']['curve'].split('_')[-1]);raw_amp=inv_predict(m['amps'][raw_g],x['F'])
    for name,pi,cm in [('true_mass',x['pi'],x['pred_cm']),('true_moments',x['pred_pi'],x['cm']),('true_b',x['pi'],x['cm'])]:
        P['raw_ablation_'+name]=alloc(raw_amp+raw_g*cover(x,coefficients(pi,cm)),bs)
    R=(RA+RB)/2;Ct=cover(x,x['b']);P['privileged_coverage']=alloc(amplitude(R,Ct,g)[:,None]/KS+g*Ct,bs);P['privileged_inverse_K']=alloc(amplitude(R,Ct,0)[:,None]/KS,bs);P['asymptotic_oracle']=alloc(tau[:,None]/KS,bs);P['oracle_A']=alloc(RA,bs);P['oracle_B']=alloc(RB,bs);P['finite_oracle']=alloc(R,bs)
    assert all(np.all((v>=KMIN)&(v<=KMAX)) and all(v[ix].sum()==len(ix)*KAVG for ix in bs) for v in P.values())
    return P,pred

def contrast(a,b):
    a=np.asarray(a,float);b=np.asarray(b,float);ma=a.mean();mb=b.mean();g=1-ma/max(mb,FLOOR);z=-(a-ma)/max(mb,FLOOR)+ma*(b-mb)/max(mb,FLOOR)**2;se=z.std(ddof=1)/np.sqrt(len(z));h=student_t.ppf(.975,len(z)-1)*se;p=2*student_t.sf(abs(g/se),len(z)-1) if se>0 else (0. if g!=0 else 1.)
    return dict(gain_pct=100*g,ci_low=100*(g-h),ci_high=100*(g+h),p=float(p))

def evaluate(name,P,pred,RE,bs,sel,valrows,x,RA,RB,errs=None,physical=None):
    folder=OUT/name;folder.mkdir(exist_ok=True);R={k:RE[np.arange(len(RE)),v-KMIN] for k,v in P.items()};base=R['cheap_selected'].mean();o=R['finite_oracle'].mean();units={k:errs[k].mean(1) if errs is not None else np.array([v[ix].mean() for ix in bs]) for k,v in R.items()};rows=[]
    for k,v in R.items():
        harm=np.array([(v[ix].mean()-R['cheap_selected'][ix].mean())/max(R['cheap_selected'][ix].mean(),FLOOR) for ix in bs]);rows.append(dict(method=k,mse=v.mean(),gain_vs_uniform_pct=100*(1-v.mean()/R['uniform'].mean()),**contrast(units[k],units['cheap_selected']),headroom_captured=(base-v.mean())/(base-o) if base>o else np.nan,harmful_batches=(harm>0).mean(),worst_decile_loss=np.sort(harm)[-max(1,math.ceil(.1*len(harm))):].mean(),at_min=(P[k]==KMIN).sum(),at_max=(P[k]==KMAX).sum()))
    df=pd.DataFrame(rows).sort_values('mse');df.to_csv(folder/'method_summary.csv',index=False);pd.DataFrame(P).to_csv(folder/'allocations.csv',index=False);valrows.to_csv(folder/'validation_candidates.csv',index=False);(folder/'selection.json').write_text(json.dumps(sel,indent=2))
    comp={k:contrast(units['structured'],units[k]) for k in ['cheap_selected','generic','no_coverage','direct_b']};(folder/'primary_contrasts.json').write_text(json.dumps(comp,indent=2))
    ablation_rows=[]
    for k in ['true_mass','true_moments','true_b']:
        for prefix,reference in [('ablation_','structured'),('raw_ablation_','structured_raw')]:
            ablation_rows.append(dict(method=prefix+k,reference=reference,**contrast(units[prefix+k],units[reference])))
    pd.DataFrame(ablation_rows).to_csv(folder/'component_ablations.csv',index=False)
    np.savez_compressed(folder/'risk_and_regions.npz',K=KS,RE=RE,RA=RA,RB=RB,true_pi=x['pi'],pred_pi=x['pred_pi'],true_cm=x['cm'],pred_cm=x['pred_cm'],true_b=x['b'],pred_b=x['pred_b'],rho=x['rho'],structured=pred[sel['structured']['curve']],generic=pred[sel['generic']['curve']],batch_id=np.concatenate([np.full(len(i),j) for j,i in enumerate(bs)])[np.argsort(np.concatenate(bs))]);np.savez_compressed(folder/'paired_units.npz',**units)
    if errs is not None:np.savez_compressed(folder/'paired_mc_errors.npz',**errs)
    if physical is not None:pd.DataFrame([dict(method=k,mean_coordinate_error=float(v.mean()),median_coordinate_error=float(np.median(v))) for k,v in physical.items()]).to_csv(folder/'localization.csv',index=False)
    def sp(a,b):return float(spearmanr(np.ravel(a),np.ravel(b)).statistic) if np.std(a)>0 and np.std(b)>0 else np.nan
    C=cover(x,x['pred_b']);Ct=cover(x,x['b']);diag=dict(mass_mae=float(abs(x['pi']-x['pred_pi']).mean()),conditional_mean_weighted_mse=float((x['pi']*((x['cm']-x['pred_cm'])**2).mean(2)).sum(1).mean()),b_spearman=sp(x['b'],x['pred_b']),coverage48_spearman=sp(C[:,48-KMIN],Ct[:,48-KMIN]),coverage_marginal24_72_spearman=sp(C[:,24-KMIN]-C[:,72-KMIN],Ct[:,24-KMIN]-Ct[:,72-KMIN]));(folder/'region_diagnostics.json').write_text(json.dumps(diag,indent=2))
    plt.figure(figsize=(10,6));z=df.sort_values('gain_pct');plt.barh(z.method,z.gain_pct);plt.axvline(0,color='black');plt.xlabel('MSE reduction vs validation-selected cheap policy (%)');plt.tight_layout();plt.savefig(folder/'gain.png',dpi=150);plt.show();plt.close();print(name,sel,comp);display(df.round(4));return df,comp


In [ ]:
assert np.allclose(coefficients(pi0,cm0),[[.09,.49]])
print("Coverage-coefficient and optimizer checks passed.")

## 3. Fresh controlled synthetic cohorts
The original Gaussian-mixture model and controlled separation/mismatch variants are reused. A separate cohort trains the regional posterior student; risk teachers, validation and final batches are independent. Exact Gaussian conditional moments are used only for offline labels and privileged diagnostics. The generator parameters are accessible to all comparators.

In [ ]:
MASTER=NEW
N=84
def rng_for(*keys):
    return np.random.default_rng(np.random.SeedSequence([MASTER,*map(int,keys)]))

def make_model(d,seed,regime,n=N):
    rng=np.random.default_rng(seed)
    c=rng.integers(0,3,n)
    modes=rng.choice(2,n,p=[.55,.45])
    latent_eps=rng.normal(size=(n,d)); obs_eps=rng.normal(size=(n,d))
    ctx=rng_for(11,d,seed)
    sep_idx=ctx.integers(0,3,n); mismatch_idx=ctx.integers(0,3,n)
    sigma=np.array([.45,1.,2.1])[c] if regime in ('original','noise_only') else np.ones(n)
    sep=np.array([.5,2.25,4.])[sep_idx] if regime in ('separation_only','combined_fixed_scale') else np.full(n,2.25)
    means=np.zeros((n,2,d)); means[:,0,0]=-sep; means[:,1,0]=sep
    x=latent_eps+means[np.arange(n),modes]+sigma[:,None]*obs_eps
    v=sigma**2/(1+sigma**2)
    a=np.array([1.,1.1,1.25])[c] if regime=='original' else np.full(n,1.1)
    u=v*a*a
    nominal=x/(1+sigma[:,None]**2)
    qmu=nominal*.88 if regime=='original' else nominal.copy()
    if regime in ('mismatch_only','combined_fixed_scale'):
        # deterministic direction perpendicular to mixture axis; offset is known to allocator
        qmu[:,1]+=np.array([0.,.5,1.5])[mismatch_idx]*np.sqrt(v)
    mismatch=np.linalg.norm(qmu-nominal,axis=1)/np.sqrt(v)
    # Exact conditional posterior; all likelihood normalizers common across components.
    lg=np.log([.55,.45])[None,:]-.5*np.sum((x[:,None,:]-means)**2,axis=2)/(1+sigma[:,None]**2)
    lg-=logsumexp(lg,axis=1,keepdims=True)
    gamma=np.exp(lg)
    pm=(x[:,None,:]+sigma[:,None,None]**2*means)/(1+sigma[:,None,None]**2)
    truth=np.sum(gamma[:,:,None]*pm,axis=1)
    post_scale=np.sqrt(d*v+np.sum(gamma*np.sum((pm-truth[:,None,:])**2,axis=2),axis=1))
    return dict(d=d,x=x,c=c,sigma=sigma,sep=sep,v=v,u=u,qmu=qmu,mismatch=mismatch,lg=lg,pm=pm,truth=truth,post_scale=post_scale)

def exact_tau2(m):
    # Closed form integral of p^2/q times squared distance to true posterior mean.
    v,u,d=m['v'],m['u'],m['d']; A=2/v-1/u
    assert np.all(A>0), 'Asymptotic variance does not exist for this proposal.'
    a=m['pm']-m['qmu'][:,None,:]; target=m['truth']-m['qmu']; terms=[]
    for j,k in itertools.product(range(2),repeat=2):
        h=(a[:,j]+a[:,k])/v[:,None]; center=h/A[:,None]
        c=(np.sum(a[:,j]**2+a[:,k]**2,axis=1)/v)-np.sum(h*h,axis=1)/A
        logc=-d*np.log(2*np.pi*v)+.5*d*np.log(2*np.pi*u)+.5*d*np.log(2*np.pi/A)-.5*c
        moment=d/A+np.sum((center-target)**2,axis=1)
        terms.append(m['lg'][:,j]+m['lg'][:,k]+logc+np.log(moment))
    t=np.exp(logsumexp(np.stack(terms,axis=1),axis=1))
    assert np.all(np.isfinite(t)&(t>0))
    return t



In [ ]:
def interval_moment(lo,hi,mean,sd):
    a=(lo-mean)/sd;b=(hi-mean)/sd
    mass=np.where(a>0,ndtr(-a)-ndtr(-b),ndtr(b)-ndtr(a));mass=np.maximum(mass,0)
    phi=lambda z:np.exp(-z*z/2)/np.sqrt(2*np.pi)
    return mass,mean*mass+sd*(phi(a)-phi(b))
def gaussian_pack(m):
    d=m['d'];x=m['x'];qmu=m['qmu'];u=m['u'];v=m['v'];pm=m['pm'];p=np.exp(m['lg']);mu=m['truth'];nom=x/(1+m['sigma'][:,None]**2)
    cuts=[[-np.inf,-2.,-1.,1.,2.,np.inf],[-np.inf,-1.5,1.5,np.inf]]
    masses=[];moments=[];rhos=[];hh=[]
    for j,k in itertools.product(range(5),range(3)):
        lows=[nom[:,0]+np.sqrt(v)*cuts[0][j],nom[:,1]+np.sqrt(v)*cuts[1][k]];highs=[nom[:,0]+np.sqrt(v)*cuts[0][j+1],nom[:,1]+np.sqrt(v)*cuts[1][k+1]]
        P=[];M=[];Q=[];QM=[]
        for axis in [0,1]:
            a,b=interval_moment(lows[axis][:,None],highs[axis][:,None],pm[:,:,axis],np.sqrt(v[:,None]));P.append(a);M.append(b)
            a,b=interval_moment(lows[axis],highs[axis],qmu[:,axis],np.sqrt(u));Q.append(a);QM.append(b)
        joint=P[0]*P[1];pi=np.sum(p*joint,1);moment=np.einsum('nj,nj,njd->nd',p,joint,pm);moment[:,0]=np.sum(p*M[0]*P[1],1);moment[:,1]=np.sum(p*M[1]*P[0],1)
        rho=Q[0]*Q[1];rhos.append(rho);masses.append(pi);moments.append(moment)
        hh.append(np.column_stack([rho,np.log(np.maximum(rho,1e-14)),QM[0]/np.maximum(Q[0],1e-14)-qmu[:,0],QM[1]/np.maximum(Q[1],1e-14)-qmu[:,1],np.full(len(x),j),np.full(len(x),k)]))
    pi=np.stack(masses,1);M=np.stack(moments,1);rho=np.stack(rhos,1);H=np.stack(hh,1);b=np.zeros_like(pi)
    assert np.allclose(pi.sum(1),1,atol=1e-10) and np.allclose(M.sum(1),mu,atol=1e-9)
    for j in range(pi.shape[1]):
        other=np.arange(pi.shape[1])!=j;outside=pi[:,other].sum(1);cm=M[:,other,:].sum(1)/np.maximum(outside[:,None],1e-300);b[:,j]=np.where(outside>1e-14,np.sum((mu-cm)**2,1)/d,0)
    F=np.column_stack([x,x*x,np.linalg.norm(x,axis=1),m['sigma'],np.sqrt(u),m['sep'],m['mismatch'],qmu])
    cm=M/np.maximum(pi[:,:,None],1e-300); cm=np.where(pi[:,:,None]>1e-14,cm,nom[:,None,:]); result=pack(F,H,rho,b);result.update(pi=pi,cm=cm); assert np.allclose(coefficients(pi,cm),b,atol=1e-7,rtol=1e-5);return result

@torch.no_grad()
def gauss_bank(m,reps,seed):
    n=len(m['x']);d=m['d'];R=np.empty((n,len(KS)));gen=torch.Generator(device=DEV).manual_seed(seed)
    for st in range(0,n,8):
        sl=slice(st,min(st+8,n));size=len(m['x'][sl]);eps=torch.randn(size,reps,KMAX,d,device=DEV,generator=gen);u=T(m['u'][sl])[:,None,None,None];v=T(m['v'][sl])[:,None,None,None];z=T(m['qmu'][sl])[:,None,None,:]+u.sqrt()*eps
        pm=T(m['pm'][sl]);lg=T(m['lg'][sl]);lp=torch.logaddexp(lg[:,0,None,None]-.5*((z-pm[:,0,None,None,:])**2).sum(-1)/v[...,0],lg[:,1,None,None]-.5*((z-pm[:,1,None,None,:])**2).sum(-1)/v[...,0]);lw=lp+.5*d*(u[...,0]/v[...,0]).log()+.5*(eps**2).sum(-1)
        den=torch.logcumsumexp(lw,2);shift=z.amin(2,keepdim=True)-1.;est=(torch.logcumsumexp(lw[...,None]+(z-shift).log(),2)-den[...,None]).exp()+shift;err=((est-T(m['truth'][sl])[:,None,None,:])**2).mean(-1);R[sl]=err[:,:,KMIN-1:].mean(1).cpu().numpy()
    assert np.isfinite(R).all();return R

def cat_models(ms):return {k:np.concatenate([m[k] for m in ms],axis=0) if isinstance(ms[0][k],np.ndarray) else ms[0][k] for k in ms[0]}
def gauss_cheap(m):
    scores={'uniform':np.ones(len(m['x'])),'context_noise':m['sigma'],'proposal_width':np.sqrt(m['u']),'component_separation':m['sep']+.05,'proposal_mismatch':m['mismatch']+.05,'combined_context':np.sqrt(m['sigma']**2+m['sep']**2+m['mismatch']**2)}
    return {k:v[:,None]**2/KS for k,v in scores.items()}


In [ ]:
SYN_ROWS=[]; CON=[]
CASES=[(4,'original'),(4,'separation_only'),(4,'mismatch_only'),(4,'combined_fixed_scale'),(8,'original'),(8,'combined_fixed_scale')]
for j,(dim,regime) in enumerate(CASES if FULL else CASES[:1]):
    seed=NEW+100000+j*10000;name=f'gaussian_{dim}D_{regime}';n=N if FULL else 12;nbatch=25 if FULL else 3;nt=1008 if FULL else 60;nv=504 if FULL else 24;ndist=4032 if FULL else 100
    ms=make_model(dim,seed,regime,ndist);mt=make_model(dim,seed+1,regime,nt);mv=make_model(dim,seed+2,regime,nv);mx=cat_models([make_model(dim,seed+100+i,regime,n) for i in range(nbatch)])
    xs,xt,xv,xx=map(gaussian_pack,[ms,mt,mv,mx]);student=train_regions(xs,xs['pi'],xs['cm'],seed+3)
    xt,xv,xx=[augment(z,*region_predict(student,z)) for z in [xt,xv,xx]]
    Rt=gauss_bank(mt,NTEACH,seed+4);Rv=gauss_bank(mv,NVAL,seed+5);mr=fit_risk(xt,Rt,seed+6)
    bv=[np.arange(i,min(i+n,nv)) for i in range(0,nv,n)];sel,vr=choose(curves(mr,xv),gauss_cheap(mv),Rv,bv);bx=[np.arange(i*n,(i+1)*n) for i in range(nbatch)]
    RA=gauss_bank(mx,NOPT,seed+7);RB=gauss_bank(mx,NOPT,seed+8);P,pred=policies(mr,xx,gauss_cheap(mx),bx,sel,RA,RB,exact_tau2(mx)/dim);RE=gauss_bank(mx,NEVAL,seed+9)
    df,cmp=evaluate(name,P,pred,RE,bx,sel,vr,xx,RA,RB);SYN_ROWS.append(df.assign(case=name));CON.extend(dict(case=name,comparator=k,**v) for k,v in cmp.items());joblib.dump(mr,OUT/name/'risk_models.joblib');net,center,scale,c0,cs,J,dd=student;torch.save(dict(state=net.cpu().state_dict(),center=center,scale=scale,c0=c0,cs=cs,J=J,d=dd),OUT/name/'region_student.pt')
    pd.concat(SYN_ROWS,ignore_index=True).to_csv(OUT/'synthetic_summary.csv',index=False)
print('Synthetic suite completed.')

## 4. UJIIndoorLoc real-data follow-up
Same survey-map construction, likelihood, calibration, proposal, official test split and partition as the preceding coverage study. A separate student learns the fitted target posterior on official fit records. It does not modify the original proposal. A coherent predicted distribution over survey states guarantees that the derived region probabilities and moments are mutually consistent. Direct prediction by this student is also evaluated as a non-sampling alternative.

In [ ]:
URLS=['https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip','https://archive.ics.uci.edu/ml/machine-learning-databases/00310/UJIIndoorLoc.zip']
def readzip(blob):
    d={}
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for n in z.namelist():
            if n.lower().endswith('.zip'): d.update(readzip(z.read(n)))
            elif n.lower().endswith('trainingdata.csv'): d['train']=pd.read_csv(io.BytesIO(z.read(n)))
            elif n.lower().endswith('validationdata.csv'): d['test']=pd.read_csv(io.BytesIO(z.read(n)))
    if not d: raise ValueError('No official CSVs found')
    return d
cache=OUT/'uji.zip'; data=None; errors=[]
for url in ([None] if cache.exists() else [])+URLS:
    try:
        if url is None: blob=cache.read_bytes()
        else:
            with urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'}),timeout=90) as r: blob=r.read()
        data=readzip(blob); assert set(data)=={'train','test'}; cache.write_bytes(blob); break
    except Exception as e: data=None;errors.append(str(e))
if data is None:
    print('Download failed:',errors,'Upload the official ZIP from https://archive.ics.uci.edu/dataset/310/ujiindoorloc')
    from google.colab import files
    blob=next(iter(files.upload().values()));data=readzip(blob);cache.write_bytes(blob)
train,test=data['train'],data['test'];sha=hashlib.sha256(blob).hexdigest();assert (len(train),len(test))==(19937,1111)
for f in (train,test): f.columns=f.columns.str.strip().str.upper()
WAPS=sorted(c for c in train if c.startswith('WAP')); A=len(WAPS); LOC=['BUILDINGID','FLOOR','LONGITUDE','LATITUDE']
assert A==(12 if TEST_FIXTURE else 520); assert not train[WAPS+LOC].isna().any().any() and not test[WAPS+LOC].isna().any().any()
print('Training / official test:',len(train),len(test),'APs:',A,'SHA256:',sha)

In [ ]:
# Select disjoint user groups using original training data only. If infeasible, explicitly disclose row split.
def split(f):
    ids=np.arange(len(f));u=f.USERID.to_numpy();g=np.unique(u);s=(f.BUILDINGID.astype(str)+'_'+f.FLOOR.astype(str)).to_numpy();rng=np.random.default_rng(SEED);best=None;bs=1e9
    for _ in range(750):
        if len(g)<8:break
        q=rng.permutation(g);a=round(.6*len(g));b=max(1,round(.13*len(g))); parts=[ids[np.isin(u,h)] for h in [q[:a],q[a:a+b],q[a+b:a+2*b],q[a+2*b:]]]; sizes=np.array([len(x) for x in parts]);score=np.sum((sizes/len(f)-[.60,.13,.13,.14])**2)
        if min(sizes)>=(20 if TEST_FIXTURE else 250) and set(s[parts[0]])==set(s) and score<bs:best=parts;bs=score
    if best is not None:return best,'user-disjoint development'
    a,b=train_test_split(ids,test_size=.4,random_state=SEED,stratify=s);c,b=train_test_split(b,train_size=1/3,random_state=SEED+1,stratify=s[b]);d,e=train_test_split(b,test_size=.5,random_state=SEED+2,stratify=s[b]);return [a,c,d,e],'stratified rows; development users can overlap'
parts,split_type=split(train); fit,cal,teacher,val=[train.iloc[p].reset_index(drop=True) for p in parts]
def cap(f,n,seed):return f.sample(n=n,random_state=seed).reset_index(drop=True) if len(f)>n else f
cal=cap(cal,1600,SEED+3);teacher=cap(teacher,1000,SEED+4);val=cap(val,512,SEED+5)
assert len(np.unique(np.concatenate(parts)))==len(train)
print(split_type, [len(x) for x in [fit,cal,teacher,val,test]])
key=fit[LOC].to_numpy(np.float64);key[:,2:]=np.round(key[:,2:],2);nodes,labels=np.unique(key,axis=0,return_inverse=True);S=len(nodes);origin=nodes[:,2:].mean(0);XY=T(nodes[:,2:]-origin);groups,gi=np.unique(nodes[:,:2],axis=0,return_inverse=True);G=len(groups);GROUP=T(np.eye(G)[gi])
print('Survey locations:',S,'building-floor groups:',G)
def encode(f):
    raw=f[WAPS].to_numpy(np.float32);d=(raw!=100).astype(np.float32);r=np.where(d>0,(raw+75)/20,0);return d,r
def arrays(f):
    d,r=encode(f);gt=f[['LONGITUDE','LATITUDE']].to_numpy(np.float64)-origin;grp=np.full(len(f),-1)
    for j,g in enumerate(groups):grp[(f.BUILDINGID.to_numpy()==g[0])&(f.FLOOR.to_numpy()==g[1])]=j
    return dict(X=T(np.concatenate([d,r,r*r],1)),inp=T(np.concatenate([d,r],1)),gt=T(gt),group=grp,phone=f.PHONEID.to_numpy(),user=f.USERID.to_numpy())
fit_a,cal_a,teach_a,val_a,test_a=map(arrays,[fit,cal,teacher,val,test])

In [ ]:
# Independent Bernoulli detection and conditional Gaussian RSSI, smoothed toward floor and global statistics.
d,r=encode(fit)
def stats(index,size):
    n=np.bincount(index,minlength=size)[:,None];c=np.zeros((size,A));s=c.copy();ss=c.copy();np.add.at(c,index,d);np.add.at(s,index,r);np.add.at(ss,index,r*r);return n,c,s,ss
n,c,s,ss=stats(labels,S);ng,cg,sg,ssg=stats(gi[labels],G);C=c.sum(0);gp=(C+.5)/(len(d)+1);gm=s.sum(0)/(C+5);gm2=(ss.sum(0)+5)/(C+5)
pg=(cg+20*gp)/(ng+20);mg=(sg+10*gm)/(cg+10);mg2=(ssg+10*gm2)/(cg+10)
p=np.clip((c+8*pg[gi])/(n+8),1e-5,1-1e-5);mu=(s+8*mg[gi])/(c+8);v=np.clip((ss+8*mg2[gi])/(c+8)-mu*mu,.15**2,9.)
base=np.log1p(-p).sum(1);coef=np.concatenate([np.log(p)-np.log1p(-p)-.5*np.log(2*np.pi*v)-.5*mu*mu/v,mu/v,-.5/v],1);BASE=T(base);COEF=T(coef)
def raw(a):return a['X']@COEF.T+BASE
with torch.no_grad():
    logits=raw(cal_a);temps=[1.,2.,4.,8.,16.,32.,64.];scores=[float(((torch.softmax(logits/h,1)@XY-cal_a['gt'])**2).sum(1).mean()) for h in temps];TEMP=temps[int(np.argmin(scores))]
d0=cal_a['X'][0,:A].cpu().numpy();r0=cal_a['X'][0,A:2*A].cpu().numpy();check=(d0*np.log(p[0])+(1-d0)*np.log1p(-p[0])+d0*(-.5*np.log(2*np.pi*v[0])-.5*(r0-mu[0])**2/v[0])).sum();assert np.isclose(float(logits[0,0]),check,atol=.02,rtol=1e-5)
print('Calibrated likelihood temperature:',TEMP)

In [ ]:
model=nn.Sequential(nn.Linear(2*A,256),nn.LayerNorm(256),nn.ReLU(),nn.Dropout(.1),nn.Linear(256,128),nn.ReLU(),nn.Linear(128,S)).to(DEV);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001);y=T(labels,torch.long)
for epoch in range(EPOCHS):
    model.train();loss_sum=0
    for ix in torch.randperm(len(y),device=DEV).split(256):
        loss=nn.functional.cross_entropy(model(fit_a['inp'][ix]),y[ix]);opt.zero_grad();loss.backward();opt.step();loss_sum+=float(loss.detach())*len(ix)
    if epoch==0 or (epoch+1)%5==0:print('Epoch',epoch+1,'CE',loss_sum/len(y))
model.eval()
with torch.no_grad():
    pc=torch.softmax(logits/TEMP,1);qc=model(cal_a['inp']);candidates=[(h,e) for h in [.75,1.,1.5,2.,3.,4.] for e in [.05,.15,.30]];sc=[float(-(pc*((1-e)*torch.softmax(qc/h,1)+e/S).log()).sum(1).mean()) for h,e in candidates];QT,EPS=candidates[int(np.argmin(sc))]
@torch.no_grad()
def proposal(a):return (1-EPS)*torch.softmax(model(a['inp'])/QT,1)+EPS/S
@torch.no_grad()
def prepare(a):
    lp=torch.log_softmax(raw(a)/TEMP,1);p=lp.exp();return dict(**a,lp=lp,p=p,q=proposal(a),ref=p@XY)
# Fit and calibration end here. Test posterior is used for offline evaluation only, never for deployable policy inputs.
teach_a,val_a,test_a=map(prepare,[teach_a,val_a,test_a]);assert torch.all(test_a['q']>0) and torch.allclose(test_a['p'].sum(1),torch.ones(len(test),device=DEV),atol=1e-5)
print('Proposal temperature and defensive mass:',QT,EPS)

In [ ]:
@torch.no_grad()
def features(a,q=None):
    q=proposal(a) if q is None else q;m=q@XY;var=(q@(XY*XY)-m*m).clamp_min(0).sum(1);ent=-(q*q.log()).sum(1);top=q.topk(2,1).values;d=a['inp'][:,:A];r=a['inp'][:,A:];count=d.sum(1);mean=r.sum(1)/count.clamp_min(1);sd=((r*r).sum(1)/count.clamp_min(1)-mean*mean).clamp_min(0).sqrt();bf=(q@GROUP).max(1).values
    return torch.stack([count,mean,sd,var.sqrt(),ent,top[:,0],top[:,0]-top[:,1],bf,(q*q).sum(1),m[:,0],m[:,1]],1).cpu().numpy()

def proxies(f):
    z={'uniform':np.ones(len(f)),'proposal_spread':f[:,3]+.1,'proposal_entropy':f[:,4]+.1,'inverse_AP_count':1/np.sqrt(f[:,0]+1),'proposal_ambiguity':1-f[:,5]+.05,'floor_uncertainty':1-f[:,7]+.05};return {k:s[:,None]**2/KS[None,:] for k,s in z.items()}

In [ ]:
# Prefix SNIS using log-cumulative sums; exact model posterior only for offline teacher/reference computation.
shift=XY.min(0).values-1;logxy=(XY-shift).log()
@torch.no_grad()
def bank(a,reps,seed,policies=None):
    gen=torch.Generator(device=DEV).manual_seed(seed);n=len(a['q']);R=np.empty((n,len(KS)),np.float64);names=list(policies or {});err={k:np.empty((reps,n),np.float32) for k in names};physical={k:np.empty((reps,n),np.float32) for k in names}
    for st in range(0,n,8):
        en=min(n,st+8);q=a['q'][st:en];m=len(q);cdf=q.cumsum(1);cdf[:,-1]=1;u=torch.rand((m,reps*KMAX),generator=gen,device=DEV);idx=torch.searchsorted(cdf.contiguous(),u.contiguous()).reshape(m,reps,KMAX).clamp_max(S-1);lw=(a['lp'][st:en]-q.log()).gather(1,idx.reshape(m,-1)).reshape(m,reps,KMAX);ld=torch.logcumsumexp(lw,2);est=torch.stack([(torch.logcumsumexp(lw+logxy[idx,j],2)-ld).exp()+shift[j] for j in range(2)],-1);e=((est-a['ref'][st:en,None,None,:])**2).sum(-1)/2;R[st:en]=e[:,:,KMIN-1:].mean(1).cpu().numpy()
        if names:
            pe=((est-a['gt'][st:en,None,None,:])**2).sum(-1).sqrt();rr=torch.arange(m,device=DEV)
            for k in names:
                kk=T(policies[k][st:en]-1,torch.long);err[k][:,st:en]=e[rr,:,kk].T.cpu().numpy();physical[k][:,st:en]=pe[rr,:,kk].T.cpu().numpy()
        if st==0 or en==n or st%400==0:print('Bank',seed, en,'/',n,flush=True)
    assert np.isfinite(R).all() and all(np.isfinite(z).all() for z in err.values());return R,err,physical
ix=T([0,1,2],torch.long);lw=T([-2.,-4.,-1.]);est=(torch.logsumexp(lw[:,None]+logxy[ix],0)-torch.logsumexp(lw,0)).exp()+shift;assert torch.allclose(est,torch.softmax(lw,0)@XY[ix],atol=.001,rtol=.0001)

In [ ]:
region=np.empty(S,int);nreg=0
for g in range(G):
    ix=np.flatnonzero(gi==g);nc=min(3,max(1,len(ix)//20));lab=KMeans(n_clusters=nc,random_state=SEED,n_init=10).fit_predict(nodes[ix,2:]) if nc>1 else np.zeros(len(ix),int)
    region[ix]=lab+nreg;nreg+=nc
REG=T(np.eye(nreg)[region]);RGEOM=(REG.T@XY)/REG.sum(0)[:,None];RSIZE=REG.sum(0)/S;RGROUP=(REG.T@GROUP).argmax(1)
assert torch.all(REG.sum(1)==1) and torch.all(REG.sum(0)>0)
@torch.no_grad()
def region_moments(p):
    p=p.double();rg=REG.double();xy=XY.double();mass=p@rg;moment=torch.stack([(p*xy[:,j])@rg for j in range(2)],2);cm=moment/mass.clamp_min(1e-300)[:,:,None];cm=torch.where(mass[:,:,None]>1e-30,cm,RGEOM.double()[None]);return mass.cpu().numpy(),cm.cpu().numpy()
@torch.no_grad()
def wifi_pack(a,q=None,offline=True):
    q=proposal(a) if q is None else q;rho=q@REG;qm=torch.stack([(q*XY[:,j])@REG for j in range(2)],2)/rho.clamp_min(1e-12)[:,:,None];qbar=q@XY;n=len(q)
    H=torch.cat([rho[:,:,None],rho.clamp_min(1e-12).log()[:,:,None],qm-qbar[:,None,:],RGEOM[None,:,:].expand(n,-1,-1),RSIZE[None,:,None].expand(n,-1,-1),nn.functional.one_hot(RGROUP,G).float()[None,:,:].expand(n,-1,-1)],2)
    # Offline posterior labels never enter online features.
    if offline:
        pi,cm=region_moments(a['lp'].double().softmax(1));b=coefficients(pi,cm)
    else:b=np.zeros((n,nreg))
    out=pack(features(a,q),H.cpu().numpy(),rho.cpu().numpy(),b)
    if offline:out.update(pi=pi,cm=cm)
    return out
print('Fixed regions:',nreg,'survey states:',S)
# Distill the target, not the proposal. Only fit-map observations supervise the student.
torch.manual_seed(NEW+300);student_wifi=nn.Sequential(nn.Linear(2*A,256),nn.LayerNorm(256),nn.SiLU(),nn.Linear(256,192),nn.SiLU(),nn.Linear(192,S)).to(DEV);opt_s=torch.optim.AdamW(student_wifi.parameters(),lr=.001,weight_decay=.0001)
with torch.no_grad():soft_fit=torch.softmax(raw(fit_a)/TEMP,1)
for ep in range(50 if FULL else 2):
    student_wifi.train();total=0.
    for ix in torch.randperm(len(soft_fit),device=DEV).split(256):
        loss=-(soft_fit[ix]*student_wifi(fit_a['inp'][ix]).log_softmax(1)).sum(1).mean();opt_s.zero_grad();loss.backward();opt_s.step();total+=float(loss.detach())*len(ix)
    if ep==0 or (ep+1)%10==0:print('Student epoch',ep+1,'cross entropy',total/len(soft_fit))
student_wifi.eval()
# Temperature of student selected against target on independent calibration records.
with torch.no_grad():
    zz=student_wifi(cal_a['inp']);pp=torch.softmax(raw(cal_a)/TEMP,1);ts=[.5,.75,1.,1.25,1.5,2.];ST=ts[int(np.argmin([float(-(pp*(zz/h).log_softmax(1)).sum(1).mean()) for h in ts]))]
@torch.no_grad()
def student_p(a):return (student_wifi(a['inp'])/ST).double().softmax(1)
@torch.no_grad()
def wifi_aug(a,q=None,offline=True):
    z=wifi_pack(a,q,offline);pi,cm=region_moments(student_p(a));return augment(z,pi,cm)
ft,fv,fx=[wifi_aug(a,a['q']) for a in [teach_a,val_a,test_a]]
# Test online independence explicitly: remove posterior and all labels.
check=wifi_aug({k:val_a[k][:8] for k in ['X','inp']},val_a['q'][:8],False)
for k in ['F','pred_pi','pred_cm','pred_b']:assert np.allclose(check[k],fv[k][:8],rtol=1e-5,atol=1e-5)
Rt,_,_=bank(teach_a,NTEACH,NEW+301);Rv,_,_=bank(val_a,NVAL,NEW+302);bv=batches(len(fv['F']),SEED+200);m=fit_risk(ft,Rt,NEW+303);sel,vr=choose(curves(m,fv),proxies(fv['F']),Rv,bv)
# Freeze selection before generating the test oracle or evaluation banks.
print('Frozen real-data selection:',sel)
bx=batches(len(fx['F']),SEED+201);RA,_,_=bank(test_a,NOPT,NEW+304);RB,_,_=bank(test_a,NOPT,NEW+305)
with torch.no_grad():tau=((test_a['p']**2/test_a['q'])*((XY[None]-test_a['ref'][:,None])**2).sum(-1)/2).sum(1).cpu().numpy()
P,pred=policies(m,fx,proxies(fx['F']),bx,sel,RA,RB,tau);RE,errs,locerr=bank(test_a,NEVAL,NEW+306,P)
with torch.no_grad():
    direct=(student_p(test_a)@XY.double()).float();locerr['exact_posterior']=(test_a['ref']-test_a['gt']).norm(dim=1).cpu().numpy();locerr['direct_student']=(direct-test_a['gt']).norm(dim=1).cpu().numpy();direct_stats=dict(student_posterior_mean_mse=float(((direct-test_a['ref'])**2).mean()),exact_posterior_mean_mse=0.,student_temperature=ST)
wifi,cmp=evaluate('wifi',P,pred,RE,bx,sel,vr,fx,RA,RB,errs,locerr);CON.extend(dict(case='wifi',comparator=k,**v) for k,v in cmp.items());dest=OUT/'wifi';(dest/'direct_student.json').write_text(json.dumps(direct_stats,indent=2));joblib.dump(m,dest/'risk_models.joblib');torch.save(student_wifi.state_dict(),dest/'posterior_student.pt');print(direct_stats)

## 5. Online runtime
Measure actual selected-state evaluations (no cached target posterior), and include student, features and allocation overhead. Exact enumeration and the direct student are essential comparators. All methods use the same fitted likelihood.

In [ ]:
@torch.no_grad()
def online_sample(a,q,k,seed):
    gen=torch.Generator(device=DEV).manual_seed(seed);n=len(k);mx=int(max(k));idx=torch.multinomial(q,mx,replacement=True,generator=gen);mask=torch.arange(mx,device=DEV)[None,:]<T(k,torch.long)[:,None];r,c=torch.where(mask);j=idx[r,c];lw=((a['X'][r]*COEF[j]).sum(1)+BASE[j])/TEMP-q[r,j].log();w=torch.full((n,mx),-torch.inf,device=DEV);w[r,c]=lw;return (torch.softmax(w,1)[:,:,None]*XY[idx]).sum(1)
ri=bx[0];a={k:test_a[k][ri] for k in ['X','inp']};b0=[np.arange(len(ri))]
def single_curve(m,x,key):
    F=x['F']
    if key=='inverse_K':return inv_predict(m['inv'],F)
    if key=='power_law':
        p=m['power'];return np.exp(np.clip(p[0].predict(F)[:,None]-np.clip(p[1].predict(F),.05,2.5)[:,None]*np.log(KS/KAVG),-30,30))
    if key=='direct_risk':
        lg=np.stack([r.predict(F) for r in m['direct']],1);return np.exp(np.clip(np.stack([np.interp(np.log(KS),np.log(KNOTS),v) for v in lg]),-30,30))
    g=float(key.split('_')[-1]);return inv_predict(m['amps'][g] if key.startswith('structured_') else m['damps'][g],F)+g*cover(x,x['pred_b'] if key.startswith('structured_') else direct_b(m,x))
@torch.no_grad()
def online(k,seed):
    if k=='exact_posterior':return torch.softmax(raw(a)/TEMP,1)@XY
    if k=='direct_student':return student_p(a)@XY.double()
    q=proposal(a)
    if k=='uniform':return online_sample(a,q,np.full(len(ri),KAVG,np.int32),seed)
    base=alloc(proxies(features(a,q))[sel['base']],b0)
    if k=='cheap_selected':counts=base
    else:
        z=sel[k]
        if z['penalty']=='fallback':counts=base
        else:
            xx=wifi_aug(a,q,False);counts=corrected(single_curve(m,xx,z['curve']),base,z['penalty'],b0)
    return online_sample(a,q,counts,seed)
rt=[]
for k in ['uniform','cheap_selected','structured','structured_raw','generic','direct_b','direct_student','exact_posterior']:
    online(k,NEW);sync()
    for j in range(15 if FULL else 2):
        sync();st=time.perf_counter();e=online(k,NEW+j+500);sync();assert torch.isfinite(e).all();rt.append(dict(method=k,repeat=j,milliseconds=1000*(time.perf_counter()-st)))
rt=pd.DataFrame(rt);rt.to_csv(dest/'runtime_trials.csv',index=False);runt=rt.groupby('method').milliseconds.agg(['median','min','max']).reset_index();runt.to_csv(dest/'runtime_summary.csv',index=False);display(runt)
plt.figure(figsize=(8,4));plt.barh(runt.method,runt['median']);plt.xlabel('Median online ms / batch');plt.tight_layout();plt.savefig(dest/'runtime.png',dpi=150);plt.show();plt.close()

## 6. Fixed decision rule and export
Holm correction across the 12 synthetic primary tests (two comparisons × six cases); the ablation is a secondary requirement, using a nominal interval. The real-data follow-up is explicitly exploratory. Headroom is measured using an estimated oracle, not a certified upper bound. A fallback cannot count as a win. No positive result is assumed.

In [ ]:
def holm(p):
    p=np.asarray(p,float);order=np.argsort(p);z=np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order]);out=np.empty_like(p);out[order]=np.minimum(z,1.);return out
contrast_df=pd.DataFrame(CON);contrast_df['holm_p']=np.nan;ix=(contrast_df.case!='wifi')&contrast_df.comparator.isin(['cheap_selected','generic']);contrast_df.loc[ix,'holm_p']=holm(contrast_df.loc[ix,'p']);contrast_df.to_csv(OUT/'all_primary_contrasts.csv',index=False)
all_results=pd.concat(SYN_ROWS+[wifi.assign(case='wifi')],ignore_index=True);all_results.to_csv(OUT/'all_method_results.csv',index=False)
gates=[]
for name,dg in all_results.groupby('case'):
    row=dg.set_index('method').loc['structured'];cs=contrast_df[contrast_df.case==name].set_index('comparator');s=json.loads((OUT/name/'selection.json').read_text());c1=cs.loc['cheap_selected'];c2=cs.loc['generic'];ab=cs.loc['no_coverage'];sig=(c1.ci_low>0 and c2.ci_low>0) and (name=='wifi' or (c1.holm_p<.05 and c2.holm_p<.05));passed=bool(c1.gain_pct>=2 and sig and ab.ci_low>0 and row.headroom_captured>=.25 and s['structured']['penalty']!='fallback');gates.append(dict(case=name,evidence='exploratory_reused_real' if name=='wifi' else 'fresh_synthetic',gain_cheap_pct=c1.gain_pct,ci_low_cheap=c1.ci_low,ci_high_cheap=c1.ci_high,holm_cheap=c1.holm_p,gain_generic_pct=c2.gain_pct,ci_low_generic=c2.ci_low,holm_generic=c2.holm_p,gain_no_coverage_pct=ab.gain_pct,ci_low_no_coverage=ab.ci_low,headroom_captured=row.headroom_captured,selected_penalty=s['structured']['penalty'],pass_gate=passed))
gate=pd.DataFrame(gates);gate.to_csv(OUT/'final_gate_summary.csv',index=False);display(gate)
manifest=dict(full=FULL,real_data=not TEST_FIXTURE,dataset='UJIIndoorLoc',sha256=sha,train_rows=len(train),test_rows=len(test),regions=nreg,states=S,split=split_type,fit_rows=len(fit),teacher_rows=len(teacher),val_rows=len(val),seed=SEED,new_seed=NEW,synthetic_cases=CASES,synthetic_test_batches=25 if FULL else 3,device=str(DEV),torch_version=torch.__version__,nopt=NOPT,neval=NEVAL,nteach=NTEACH,nval=NVAL,epochs=EPOCHS,budget_avg=KAVG,kmin=KMIN,kmax=KMAX,real_test_reused=True,real_ci='conditional Monte Carlo on fixed records',synthetic_ci='paired independent batch-level',elapsed_seconds=time.perf_counter()-start_total,gate='Holm primary synthetic contrasts; real exploratory',coverage_gamma=GAMMAS,penalties=PENALTIES,student_temperature=ST)
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2));torch.save(model.cpu().state_dict(),dest/'proposal.pt');np.savez_compressed(dest/'target_model.npz',nodes=nodes,origin=origin,coef=coef,base=base,temperature=TEMP,proposal_temperature=QT,defensive_mass=EPS,region=region);np.savez_compressed(dest/'development_split_indices.npz',fit=parts[0],cal=parts[1],teacher=parts[2],validation=parts[3])
archive=OUT.parent/('Final_Structured_Coverage_SNIS_Results.zip' if FULL and not TEST_FIXTURE else 'Final_Structured_Coverage_DEBUG.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file() and f.name!='uji.zip':z.write(f,f.relative_to(OUT))
print('Saved:',archive)
if not TEST_FIXTURE:
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:pass